# Tiếp tục huấn luyện từ Checkpoint (Resume training from checkpoint)

Notebook này sử dụng để TIẾP TỤC huấn luyện mô hình 3D Gaussian Splatting từ một checkpoint có sẵn (ví dụ từ 10k lên 15k, hoặc từ 20k lên 30k). Quá trình render, suy luận và tạo file nộp bài được xử lý riêng trong notebook inference.


## 1. Cấu hình huấn luyện

Khai báo đường dẫn dataset, thư mục đầu ra, cấu hình tiếp tục huấn luyện, phân bổ GPU và các siêu tham số của 3D Gaussian Splatting.

In [ ]:
import os
from pathlib import Path

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

DATASET_ROOT = Path('/kaggle/input/datasets/dangthtai/bts-digital-twin-dataset')
WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'gaussian-splatting'
OUTPUT_ROOT = WORK_DIR / 'outputs'
CONVERTED_ROOT = WORK_DIR / 'converted_datasets'

GIT_REPO_URL = 'https://github.com/graphdeco-inria/gaussian-splatting.git'

TRAIN_CONFIG = {
    # Dataset split và danh sách scene cần train. None sẽ chọn tất cả scene.
    'train_splits': ['public_set'],
    'target_scenes': ['HCM0204'],

    # Tổng số vòng lặp tối ưu của model.
    'iterations': 30000,  # ANTIGRAVITY: Giảm xuống 15000 để train nhanh
    # Các iteration chạy evaluation; -1 vô hiệu hóa evaluation định kỳ.
    'test_iterations': [-1],
    # Lưu point cloud/model hoàn chỉnh tại các iteration này.
    'save_iterations': [30000],
    # Lưu trạng thái optimizer để có thể tiếp tục training.
    'checkpoint_iterations': list(range(2000, 30001, 2000)),

    # Tiếp tục từ checkpoint 20k thay vì train lại từ đầu.
    'resume_iteration': 10000,  # ANTIGRAVITY: Mốc checkpoint muốn resume (ví dụ 10000 hoặc 20000)
    # Tự tìm và sao chép checkpoint từ Kaggle Input khi cần.
    'auto_resume': True,  # ANTIGRAVITY: Bật tự động tìm kiếm và sao chép checkpoint từ Kaggle Input
    # Kernel train chứa checkpoint để tải về khi cần resume (dùng Kaggle API)
    'train_kernel_slug': 'ptquanh/b-i-1-train-c-nh-n',

    # 1 giữ nguyên độ phân giải ảnh; giá trị 2, 4, 8 sẽ downsample tương ứng.
    'resolution': 1,
    # Giữ ảnh nguồn trên CPU để giảm mức sử dụng VRAM.
    'data_device': 'cpu',
    # Chạy bước undistortion COLMAP bằng CPU thay vì GPU.
    'no_gpu_colmap': True,

    # GPU khả dụng cho các job; với một job song song, GPU 0 được sử dụng.
    'gpu_ids': [0, 1],
    # Giới hạn số scene được train đồng thời để tránh thiếu VRAM.
    'max_parallel_train_jobs': 1,
    # True sẽ bỏ qua scene đã có point_cloud ở iteration cuối.
    'skip_trained': False,

    # Bật/tắt antialiasing trong rasterizer khi render ảnh train.
    'use_antialiasing': True,
    # Sparse Adam chỉ cập nhật các Gaussian tham gia vào ảnh hiện tại.
    'optimizer_type': 'sparse_adam',

    # Số bước dùng để giảm learning rate của vị trí Gaussian về giá trị cuối.
    'position_lr_max_steps': 15000,
    # Trọng số SSIM trong loss; phần còn lại là L1 loss.
    'lambda_dssim': 0.25,
    # Ngưỡng kích thước theo scene extent để quyết định clone hay split Gaussian.
    'percent_dense': 0.005,  # ANTIGRAVITY: Khôi phục về 0.005 tránh bùng nổ số điểm gây OOM

    # Bắt đầu tạo và tách Gaussian sau iteration này.
    'densify_from_iter': 500,
    # Thực hiện densification sau mỗi 100 iterations.
    'densification_interval': 100,
    # Chỉ densify Gaussian có gradient vị trí đạt ngưỡng này.
    'densify_grad_threshold': 0.00010,  # ANTIGRAVITY: Khôi phục về 0.00010 để tránh OOM trên T4 GPU (STT 2-7 dùng 0.0002)
    # Dừng densification để các Gaussian mới ổn định trong 5k bước cuối.
    # Resume full-resolution training with controlled densification to limit VRAM growth.
    'densify_until_iter': 12000,  # ANTIGRAVITY: Đặt dừng densify ở 12k (tương ứng đuôi hội tụ 3k)
    # Đặt lại opacity định kỳ để loại bỏ Gaussian dư thừa và tránh bão hòa.
    'opacity_reset_interval': 2000,  # ANTIGRAVITY: Giảm xuống 2000 để dọn dẹp các Gaussian rác sớm hơn, tránh tràn VRAM
    # Keep the resumed model unchanged; checkpoint tensors are released after restore instead.
    'resume_prune_fraction': 0.0,
}

TRAIN_SPLITS = TRAIN_CONFIG['train_splits']
TARGET_SCENES = TRAIN_CONFIG['target_scenes']
ITERATIONS = TRAIN_CONFIG['iterations']
TEST_ITERATIONS = TRAIN_CONFIG['test_iterations']
SAVE_ITERATIONS = TRAIN_CONFIG['save_iterations']
CHECKPOINT_ITERATIONS = TRAIN_CONFIG['checkpoint_iterations']
AUTO_RESUME = TRAIN_CONFIG['auto_resume']
RESUME_ITERATION = TRAIN_CONFIG.get('resume_iteration')
RESOLUTION = TRAIN_CONFIG['resolution']
DATA_DEVICE = TRAIN_CONFIG['data_device']
NO_GPU_COLMAP = TRAIN_CONFIG['no_gpu_colmap']
GPU_IDS = TRAIN_CONFIG['gpu_ids']
MAX_PARALLEL_TRAIN_JOBS = TRAIN_CONFIG['max_parallel_train_jobs']
SKIP_TRAINED = TRAIN_CONFIG['skip_trained']
USE_ANTIALIASING = TRAIN_CONFIG['use_antialiasing']
OPTIMIZER_TYPE = TRAIN_CONFIG['optimizer_type']
POSITION_LR_MAX_STEPS = TRAIN_CONFIG['position_lr_max_steps']
LAMBDA_DSSIM = TRAIN_CONFIG['lambda_dssim']
PERCENT_DENSE = TRAIN_CONFIG['percent_dense']
DENSIFY_FROM_ITER = TRAIN_CONFIG['densify_from_iter']
DENSIFICATION_INTERVAL = TRAIN_CONFIG['densification_interval']
DENSIFY_GRAD_THRESHOLD = TRAIN_CONFIG['densify_grad_threshold']
DENSIFY_UNTIL_ITER = TRAIN_CONFIG['densify_until_iter']
OPACITY_RESET_INTERVAL = TRAIN_CONFIG['opacity_reset_interval']
RESUME_PRUNE_FRACTION = TRAIN_CONFIG['resume_prune_fraction']
TRAIN_KERNEL_SLUG = TRAIN_CONFIG.get('train_kernel_slug', 'ptquanh/b-i-1-train-public-scene')

print('Dataset:', {
    'root': DATASET_ROOT,
    'splits': TRAIN_SPLITS,
    'scenes': TARGET_SCENES or 'ALL',
})
print('Training schedule:', {
    'iterations': ITERATIONS,
    'test_iterations': TEST_ITERATIONS,
    'save_iterations': SAVE_ITERATIONS,
    'checkpoint_iterations': CHECKPOINT_ITERATIONS,
    'resume_iteration': RESUME_ITERATION if AUTO_RESUME else None,
})
print('Quality:', {
    'position_lr_max_steps': POSITION_LR_MAX_STEPS,
    'lambda_dssim': LAMBDA_DSSIM,
    'percent_dense': PERCENT_DENSE,
    'densify_from_iter': DENSIFY_FROM_ITER,
    'densification_interval': DENSIFICATION_INTERVAL,
    'densify_grad_threshold': DENSIFY_GRAD_THRESHOLD,
    'densify_until_iter': DENSIFY_UNTIL_ITER,
    'opacity_reset_interval': OPACITY_RESET_INTERVAL,
})
print('Runtime:', {
    'resolution': RESOLUTION,
    'data_device': DATA_DEVICE,
    'gpu_ids': GPU_IDS,
    'max_parallel_jobs': MAX_PARALLEL_TRAIN_JOBS,
    'optimizer': OPTIMIZER_TYPE,
    'antialiasing': USE_ANTIALIASING,
    'no_gpu_colmap': NO_GPU_COLMAP,
    'skip_trained': SKIP_TRAINED,
    'pytorch_alloc_conf': os.environ.get('PYTORCH_ALLOC_CONF'),
})


## 2. Các dependency hệ thống trên Kaggle

Kiểm tra GPU khả dụng, sau đó cài đặt COLMAP và các công cụ cần thiết để biên dịch mã nguồn.

In [ ]:
!nvidia-smi
!apt-get update -qq
!apt-get install -y -qq colmap ninja-build

## 3. Repository Gaussian Splatting

Clone repository gốc và chỉ khởi tạo các submodule cần thiết cho quá trình huấn luyện.

In [ ]:
import shutil
import subprocess

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', GIT_REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository')
else:
    print('Repo exists:', REPO_DIR)

def init_required_submodule(path, fallback_url=None, recursive=False):
    target = REPO_DIR / path
    command = ['git', 'submodule', 'update', '--init']
    if recursive:
        command.append('--recursive')
    command.append(path)
    try:
        subprocess.check_call(command, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        if fallback_url is None:
            raise
        print(f'Submodule {path} failed from .gitmodules; cloning fallback {fallback_url}')
        shutil.rmtree(target, ignore_errors=True)
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['git', 'clone', fallback_url, str(target)])

# Only these Python/CUDA extensions are needed for training. SIBR_viewers is a GUI viewer
# and is intentionally skipped because its GitLab submodule is often unavailable on Kaggle.
init_required_submodule('submodules/diff-gaussian-rasterization', recursive=True)
init_required_submodule('submodules/fused-ssim')
init_required_submodule('submodules/simple-knn', 'https://github.com/camenduru/simple-knn.git')

os.chdir(REPO_DIR)
print('cwd:', Path.cwd())



## 4. Các extension Python và CUDA

Cài đặt các package Python, biên dịch CUDA extension và kiểm tra khả năng hỗ trợ Sparse Adam.

In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel ninja
!python -m pip install plyfile tqdm opencv-python joblib pillow

import sys
from importlib import import_module
import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('nvcc:', shutil.which('nvcc'))
subprocess.run(['nvcc', '--version'], check=False)

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available. Enable GPU accelerator before running this notebook.')

# Use the accelerated rasterizer branch required by --optimizer_type sparse_adam.
# If this branch cannot be fetched, the later import check will fail clearly.
diff_rast = Path('submodules/diff-gaussian-rasterization')
subprocess.run(['git', 'fetch', 'origin', '3dgs_accel'], cwd=diff_rast, check=True)
subprocess.run(['git', 'checkout', '3dgs_accel'], cwd=diff_rast, check=True)
shutil.rmtree(diff_rast / 'build', ignore_errors=True)

!python -m pip uninstall -y diff-gaussian-rasterization
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/diff-gaussian-rasterization
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/simple-knn
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/fused-ssim

from diff_gaussian_rasterization import SparseGaussianAdam
import_module('simple_knn._C')
import_module('fused_ssim')
print('CUDA extensions import OK')
print('SparseGaussianAdam import OK:', SparseGaussianAdam)


## 5. Tìm kiếm và chuẩn bị dataset

Tìm các scene cần huấn luyện, kiểm tra file và camera pose, đồng thời chuyển đổi những camera model chưa được hỗ trợ khi cần.

In [ ]:
import csv
import struct
from PIL import Image, ImageOps, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

CAMERA_MODELS = {
    0: ('SIMPLE_PINHOLE', 3), 1: ('PINHOLE', 4), 2: ('SIMPLE_RADIAL', 4),
    3: ('RADIAL', 5), 4: ('OPENCV', 8), 5: ('OPENCV_FISHEYE', 8),
    6: ('FULL_OPENCV', 12), 7: ('FOV', 5), 8: ('SIMPLE_RADIAL_FISHEYE', 4),
    9: ('RADIAL_FISHEYE', 5), 10: ('THIN_PRISM_FISHEYE', 12),
}

def read_camera_models(scene_path):
    cameras_bin = Path(scene_path) / 'sparse' / '0' / 'cameras.bin'
    models = []
    with cameras_bin.open('rb') as f:
        num = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num):
            camera_id, model_id, width, height = struct.unpack('<iiQQ', f.read(24))
            name, nparams = CAMERA_MODELS[model_id]
            params = struct.unpack('<' + 'd' * nparams, f.read(8 * nparams))
            models.append({'id': camera_id, 'model': name, 'width': width, 'height': height, 'params': params})
    return models

def prune_colmap_images_to_existing_files(scene_path):
    scene_path = Path(scene_path)
    images_bin = scene_path / 'sparse' / '0' / 'images.bin'
    images_dir = scene_path / 'images'
    existing = {p.name for p in images_dir.iterdir() if p.is_file()}
    kept, removed = [], []
    with images_bin.open('rb') as f:
        num_images = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_images):
            fixed = f.read(64)
            name_bytes = bytearray()
            while True:
                ch = f.read(1)
                if ch == b'':
                    raise EOFError('Unexpected EOF while reading COLMAP image name')
                name_bytes += ch
                if ch == b'\x00':
                    break
            name = name_bytes[:-1].decode('utf-8')
            npoints_bytes = f.read(8)
            npoints = struct.unpack('<Q', npoints_bytes)[0]
            points_bytes = f.read(24 * npoints)
            record = fixed + bytes(name_bytes) + npoints_bytes + points_bytes
            if Path(name).name in existing:
                kept.append(record)
            else:
                removed.append(name)
    if removed:
        backup = images_bin.with_suffix('.bin.before_prune')
        if not backup.exists():
            shutil.copy2(images_bin, backup)
        with images_bin.open('wb') as f:
            f.write(struct.pack('<Q', len(kept)))
            for record in kept:
                f.write(record)
        print(f'Pruned images.bin: kept {len(kept)}/{len(kept) + len(removed)}, removed {len(removed)}')
    return len(kept), len(removed)

def find_challenge_scenes(root, splits):
    root = Path(root)
    scenes = []
    for split in splits:
        for test_csv in sorted(root.rglob(f'phase1/{split}/*/test/test_poses.csv')):
            scene_dir = test_csv.parents[1]
            train_dir = scene_dir / 'train'
            if (train_dir / 'images').exists() and (train_dir / 'sparse' / '0' / 'cameras.bin').exists():
                scenes.append({
                    'split': split,
                    'scene_name': scene_dir.name,
                    'train_dir': train_dir,
                    'test_csv': test_csv,
                })
    return scenes

def read_colmap_image_names(scene_path):
    images_bin = Path(scene_path) / 'sparse' / '0' / 'images.bin'
    names = []
    with images_bin.open('rb') as f:
        num_images = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_images):
            f.read(64)
            name_bytes = bytearray()
            while True:
                ch = f.read(1)
                if ch == b'':
                    raise EOFError('Unexpected EOF while reading COLMAP image name')
                if ch == b'\x00':
                    break
                name_bytes += ch
            npoints = struct.unpack('<Q', f.read(8))[0]
            f.seek(24 * npoints, 1)
            names.append(Path(name_bytes.decode('utf-8')).name)
    return names

def validate_scene_file_consistency(scene):
    train_dir = Path(scene['train_dir'])
    test_csv = Path(scene['test_csv'])
    test_images = test_csv.parent / 'images'
    train_files = {p.name for p in (train_dir / 'images').iterdir() if p.is_file()}
    test_files = {p.name for p in test_images.iterdir() if p.is_file()} if test_images.exists() else set()
    colmap_names = set(read_colmap_image_names(train_dir))
    missing_train = sorted(colmap_names - train_files)
    unregistered_train = sorted(train_files - colmap_names)
    with test_csv.open(newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    required_cols = {'image_name', 'qw', 'qx', 'qy', 'qz', 'tx', 'ty', 'tz', 'fx', 'fy', 'cx', 'cy', 'width', 'height'}
    missing_cols = required_cols - set(rows[0].keys() if rows else [])
    if missing_cols:
        raise RuntimeError(f"{scene['scene_name']}: test_poses.csv missing columns: {sorted(missing_cols)}")
    pose_names = {Path(row['image_name']).name for row in rows}
    missing_test = sorted(pose_names - test_files) if test_images.exists() else []
    extra_test = sorted(test_files - pose_names) if test_images.exists() else []
    test_files_label = len(test_files) if test_images.exists() else 'missing/optional'
    print(f"Check {scene['split']}/{scene['scene_name']}: train_files={len(train_files)}, colmap_records={len(colmap_names)}, test_files={test_files_label}, test_poses={len(rows)}")
    if missing_train:
        print(f"  WARN: COLMAP references {len(missing_train)} images not in train/images; prepare_scene_for_training() will prune them.")
    if unregistered_train:
        raise RuntimeError(f"{scene['scene_name']}: train/images has files not registered in COLMAP: {unregistered_train[:10]}")
    if not test_images.exists():
        print('  INFO: test/images not found; this is OK for private scenes because rendering uses test_poses.csv.')
    if missing_test:
        raise RuntimeError(f"{scene['scene_name']}: test_poses.csv references missing test images: {missing_test[:10]}")
    if extra_test:
        raise RuntimeError(f"{scene['scene_name']}: test/images has files not in test_poses.csv: {extra_test[:10]}")

def reencode_images_for_colmap(src_images, dst_input):
    dst_input.mkdir(parents=True, exist_ok=True)
    files = [p for p in Path(src_images).iterdir() if p.is_file()]
    print('Re-encoding images:', len(files))
    bad = []
    for idx, src in enumerate(files, 1):
        try:
            with Image.open(src) as im:
                im = ImageOps.exif_transpose(im).convert('RGB')
                im.save(dst_input / src.name, format='JPEG', quality=95, optimize=True)
        except Exception as exc:
            bad.append((src, exc))
        if idx % 50 == 0:
            print(f'  {idx}/{len(files)}')
    if bad:
        for src, exc in bad[:20]:
            print('Bad image:', src, repr(exc))
        raise RuntimeError(f'{len(bad)} images cannot be decoded')

def prepare_scene_for_training(scene):
    train_dir = Path(scene['train_dir'])
    model_names = {m['model'] for m in read_camera_models(train_dir)}
    if model_names <= {'PINHOLE', 'SIMPLE_PINHOLE'}:
        prune_colmap_images_to_existing_files(train_dir)
        return train_dir

    converted = CONVERTED_ROOT / scene['split'] / scene['scene_name'] / 'train'
    if (converted / 'images').exists() and (converted / 'sparse' / '0' / 'cameras.bin').exists():
        converted_models = {m['model'] for m in read_camera_models(converted)}
        if converted_models <= {'PINHOLE', 'SIMPLE_PINHOLE'}:
            print('Use converted:', converted)
            prune_colmap_images_to_existing_files(converted)
            return converted
        shutil.rmtree(converted)
    elif converted.exists():
        shutil.rmtree(converted)

    print('Convert scene:', scene['split'], scene['scene_name'])
    converted.mkdir(parents=True, exist_ok=True)
    reencode_images_for_colmap(train_dir / 'images', converted / 'input')
    shutil.copytree(train_dir / 'sparse', converted / 'distorted' / 'sparse', dirs_exist_ok=True)
    command = [
        'python', 'convert.py', '-s', str(converted), '--skip_matching'
    ]
    if NO_GPU_COLMAP:
        command.append('--no_gpu')
    print('Running:', ' '.join(command))
    subprocess.check_call(command, cwd=REPO_DIR)
    prune_colmap_images_to_existing_files(converted)
    return converted

all_train_scenes = find_challenge_scenes(DATASET_ROOT, TRAIN_SPLITS)
if TARGET_SCENES is not None:
    all_train_scenes = [s for s in all_train_scenes if s['scene_name'] in TARGET_SCENES]
    found = {s['scene_name'] for s in all_train_scenes}
    missing = sorted(set(TARGET_SCENES) - found)
    if missing:
        raise RuntimeError(f'Missing target scenes: {missing}')
if not all_train_scenes:
    raise RuntimeError(f'No training scenes found for splits: {TRAIN_SPLITS}')
for scene in all_train_scenes:
    validate_scene_file_consistency(scene)
print('Train scenes:', [(s['split'], s['scene_name']) for s in all_train_scenes])


## 6. Tiếp tục và thực hiện huấn luyện

Khôi phục checkpoint tin cậy, tạo lệnh huấn luyện, chạy từng scene và lưu log để phục vụ chẩn đoán lỗi.

In [ ]:
# Repeat the PyTorch 2.6 compatibility check here because users commonly rerun only
# this cell after restoring a checkpoint. Checkpoints are trusted notebook outputs.
train_py = REPO_DIR / 'train.py'
train_source = train_py.read_text(encoding='utf-8')
legacy_load = 'torch.load(checkpoint)'
trusted_load = 'torch.load(checkpoint, weights_only=False)'
if legacy_load in train_source:
    train_py.write_text(train_source.replace(legacy_load, trusted_load), encoding='utf-8')
    print('Patched train.py for trusted PyTorch 2.6 checkpoint loading')
elif trusted_load not in train_source:
    raise RuntimeError('Could not verify the torch.load checkpoint compatibility patch in train.py')

# Release the deserialized checkpoint container after Gaussian parameters and optimizer
# state have been restored. This avoids retaining resume-only CUDA allocations.
train_source = train_py.read_text(encoding='utf-8')
cleanup_marker = '# CODEX_RELEASE_RESTORED_CHECKPOINT'
restore_line = '        gaussians.restore(model_params, opt)'
if cleanup_marker not in train_source:
    if restore_line not in train_source:
        raise RuntimeError('Could not locate checkpoint restore point in train.py')
    cleanup_patch = restore_line + '''
        # CODEX_RELEASE_RESTORED_CHECKPOINT
        del model_params
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        print("Released restored checkpoint tensors from CUDA cache")
'''
    train_py.write_text(train_source.replace(restore_line, cleanup_patch, 1), encoding='utf-8')
    print('Patched train.py to release restored checkpoint tensors')

    # ANTIGRAVITY: Seed fixation patch to eliminate grading variance
    train_source = train_py.read_text(encoding='utf-8')
    seed_marker = '# ANTIGRAVITY_FIX_SEED'
    if seed_marker not in train_source:
        main_entry = 'if __name__ == "__main__":'
        if main_entry not in train_source:
            raise RuntimeError('Could not locate __main__ entry point in train.py')
        seed_patch = main_entry + '''
    # ANTIGRAVITY_FIX_SEED
    import random
    import numpy as np
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
'''
        train_py.write_text(train_source.replace(main_entry, seed_patch, 1), encoding='utf-8')
        print('Patched train.py to enforce deterministic training seeds')

# ANTIGRAVITY: Auto-delete old checkpoints patch
    train_source = train_py.read_text(encoding='utf-8')
    target_save_line = 'torch.save((scene.gaussians.capture(), iteration), scene.model_path + "/chkpnt" + str(iteration) + ".pth")'
    
    if target_save_line in train_source and '# Auto-delete old checkpoints' not in train_source:
        cleanup_patch = target_save_line + '''
                # Auto-delete old checkpoints to save Kaggle disk space
                try:
                    import glob, os
                    chkpnts = glob.glob(scene.model_path + "/chkpnt*.pth")
                    if len(chkpnts) > 1:
                        chkpnts.sort(key=os.path.getmtime)
                        for old_file in chkpnts[:-1]:
                            os.remove(old_file)
                            print(f"Removed old checkpoint to save space: {os.path.basename(old_file)}")
                except Exception as e:
                    print(f"Error removing old checkpoint: {e}")
'''
        train_py.write_text(train_source.replace(target_save_line, cleanup_patch, 1), encoding='utf-8')
        print('Patched train.py to auto-delete old checkpoints')
    elif target_save_line not in train_source:
        print('Warning: Could not find checkpoint save line in train.py to patch auto-delete.')

def checkpoint_iteration(path):
    stem = Path(path).stem
    if not stem.startswith('chkpnt'):
        return None
    try:
        return int(stem.removeprefix('chkpnt'))
    except ValueError:
        return None

def seed_resume_checkpoint(model_path):
    if not AUTO_RESUME or RESUME_ITERATION is None:
        return
    model_path = Path(model_path)
    destination = model_path / f'chkpnt{RESUME_ITERATION}.pth'
    if destination.exists():
        return
    
    # 1. Thử tìm trong các Kaggle Input đính kèm trước
    matches = list(Path('/kaggle/input').rglob(f'chkpnt{RESUME_ITERATION}.pth'))
    scene_name = model_path.name
    scene_matches = [m for m in matches if scene_name in m.parts]
    if not scene_matches and matches:
        scene_matches = matches # Fallback nếu không khớp scene_name trong path
        
    if len(scene_matches) == 1:
        model_path.mkdir(parents=True, exist_ok=True)
        shutil.copy2(scene_matches[0], destination)
        print(f'Seeded resume checkpoint from Kaggle Input: {scene_matches[0]} -> {destination}')
        return
    elif len(scene_matches) > 1:
        print(f'Warning: Found multiple checkpoints in /kaggle/input: {scene_matches}. Using the first one.')
        model_path.mkdir(parents=True, exist_ok=True)
        shutil.copy2(scene_matches[0], destination)
        print(f'Seeded resume checkpoint from Kaggle Input: {scene_matches[0]} -> {destination}')
        return
        
    # 2. Nếu không tìm thấy trong Input, tự động tải qua Kaggle API
    print(f'Checkpoint not found in Kaggle Input. Trying to download using Kaggle CLI from kernel: {TRAIN_KERNEL_SLUG}...')
    kaggle_cli = shutil.which('kaggle')
    if kaggle_cli is None:
        raise RuntimeError(
            f'Expected chkpnt{RESUME_ITERATION}.pth in /kaggle/input, or a configured Kaggle API to download. '
            'Please attach the train kernel output or upload kaggle.json API key.'
        )
        
    temp_download_dir = WORK_DIR / 'temp_checkpoint_download'
    temp_download_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [kaggle_cli, 'kernels', 'output', TRAIN_KERNEL_SLUG, '-p', str(temp_download_dir)]
    print('Running download command:', ' '.join(cmd))
    
    import subprocess
    subprocess.check_call(cmd)
    
    # Tìm kiếm checkpoint trong thư mục vừa tải về
    downloaded_matches = list(temp_download_dir.rglob(f'chkpnt{RESUME_ITERATION}.pth'))
    scene_downloaded_matches = [m for m in downloaded_matches if scene_name in m.parts]
    if not scene_downloaded_matches and downloaded_matches:
        scene_downloaded_matches = downloaded_matches
        
    if not scene_downloaded_matches:
        shutil.rmtree(temp_download_dir, ignore_errors=True)
        raise FileNotFoundError(f'Could not find chkpnt{RESUME_ITERATION}.pth for scene {scene_name} in downloaded outputs.')
        
    model_path.mkdir(parents=True, exist_ok=True)
    shutil.copy2(scene_downloaded_matches[0], destination)
    print(f'Successfully downloaded and seeded checkpoint: {scene_downloaded_matches[0]} -> {destination}')
    
    # Dọn dẹp thư mục tạm
    shutil.rmtree(temp_download_dir, ignore_errors=True)

def latest_resume_checkpoint(model_path):
    candidates = []
    for path in Path(model_path).glob('chkpnt*.pth'):
        iteration = checkpoint_iteration(path)
        if iteration is not None and iteration < ITERATIONS and (RESUME_ITERATION is None or iteration == RESUME_ITERATION):
            candidates.append((iteration, path))
    if not candidates:
        return None, None
    return max(candidates, key=lambda item: item[0])

def scene_key(scene):
    return scene['split'], scene['scene_name']

def model_path_for(scene):
    return OUTPUT_ROOT / scene['split'] / scene['scene_name']

def trained_model_marker(model_path):
    return (
        Path(model_path)
        / 'point_cloud'
        / f'iteration_{ITERATIONS}'
        / 'point_cloud.ply'
    )

def append_optional_argument(command, name, value):
    if value is not None:
        command.extend([name, str(value)])

def build_train_command(train_scene, model_path, attempt, resume_checkpoint):
    command = [
        'python',
        'train.py',
        '-s',
        str(train_scene),
        '-m',
        str(model_path),
        '--iterations',
        str(ITERATIONS),
        '--resolution',
        str(RESOLUTION),
        '--data_device',
        DATA_DEVICE,
        '--test_iterations',
        *map(str, TEST_ITERATIONS),
        '--save_iterations',
        *map(str, SAVE_ITERATIONS),
        '--optimizer_type',
        attempt['optimizer_type'],
        '--disable_viewer',
    ]
    optional_arguments = {
        '--densify_grad_threshold': DENSIFY_GRAD_THRESHOLD,
        '--densify_from_iter': DENSIFY_FROM_ITER,
        '--densification_interval': DENSIFICATION_INTERVAL,
        '--densify_until_iter': DENSIFY_UNTIL_ITER,
        '--position_lr_max_steps': POSITION_LR_MAX_STEPS,
        '--lambda_dssim': LAMBDA_DSSIM,
        '--percent_dense': PERCENT_DENSE,
        '--opacity_reset_interval': OPACITY_RESET_INTERVAL,
    }
    for name, value in optional_arguments.items():
        append_optional_argument(command, name, value)
    if CHECKPOINT_ITERATIONS:
        command.extend([
            '--checkpoint_iterations',
            *map(str, CHECKPOINT_ITERATIONS),
        ])
    if resume_checkpoint is not None:
        command.extend(['--start_checkpoint', str(resume_checkpoint)])
    if attempt['antialiasing']:
        command.append('--antialiasing')
    return command

def training_environment(gpu_id):
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    environment['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
    environment['RESUME_PRUNE_FRACTION'] = str(RESUME_PRUNE_FRACTION)
    return environment

TRAIN_ATTEMPTS = [
    {'name': 'sparse_adam' + ('_aa' if USE_ANTIALIASING else '_noaa'), 'optimizer_type': OPTIMIZER_TYPE, 'antialiasing': USE_ANTIALIASING},
    {'name': 'default_aa', 'optimizer_type': 'default', 'antialiasing': True},
    {'name': 'default_noaa', 'optimizer_type': 'default', 'antialiasing': False},
]

# Remove duplicate attempts while preserving order.
_seen_attempts = set()
TRAIN_ATTEMPTS = [
    attempt for attempt in TRAIN_ATTEMPTS
    if not ((attempt['optimizer_type'], attempt['antialiasing']) in _seen_attempts
            or _seen_attempts.add((attempt['optimizer_type'], attempt['antialiasing'])))
]

prepared_scene_paths = {}
trained_scene_settings = {}

train_queue = []
for scene in all_train_scenes:
    key = scene_key(scene)
    print('\n' + '=' * 80)
    print('Preparing:', key)
    model_path = model_path_for(scene)
    marker = trained_model_marker(model_path)
    if SKIP_TRAINED and marker.exists():
        print('Skip trained scene:', marker)
        trained_scene_settings[key] = {'optimizer_type': OPTIMIZER_TYPE, 'antialiasing': USE_ANTIALIASING, 'skipped': True}
        continue
    train_scene = prepare_scene_for_training(scene)
    prepared_scene_paths[key] = train_scene
    train_queue.append({'scene': scene, 'attempt_idx': 0})

failed_scenes = {}
while train_queue:
    batch = train_queue[:MAX_PARALLEL_TRAIN_JOBS]
    train_queue = train_queue[MAX_PARALLEL_TRAIN_JOBS:]
    running = []

    for slot, state in enumerate(batch):
        scene = state['scene']
        attempt = TRAIN_ATTEMPTS[state['attempt_idx']]
        key = scene_key(scene)
        gpu_id = GPU_IDS[slot % len(GPU_IDS)]
        model_path = model_path_for(scene)
        train_scene = prepared_scene_paths[key]

        seed_resume_checkpoint(model_path)
        resume_iter, resume_checkpoint = (None, None)
        if AUTO_RESUME and state['attempt_idx'] == 0:
            resume_iter, resume_checkpoint = latest_resume_checkpoint(model_path)

        if AUTO_RESUME and resume_checkpoint is None:
            raise FileNotFoundError(f'Resume requested but chkpnt{RESUME_ITERATION}.pth was not found in {model_path}')
        if model_path.exists() and resume_checkpoint is None:
            shutil.rmtree(model_path)
        elif resume_checkpoint is not None:
            print(f"Resume {key} from checkpoint iteration {resume_iter}: {resume_checkpoint}")
        command = build_train_command(
            train_scene,
            model_path,
            attempt,
            resume_checkpoint,
        )
        environment = training_environment(gpu_id)
        resume_suffix = f"_resume_{resume_iter}" if resume_checkpoint is not None else ''
        log_path = model_path / f"train_{attempt['name']}{resume_suffix}.log"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Launching {key} attempt={attempt['name']} on GPU {gpu_id}")
        print('Running:', ' '.join(map(str, command)))
        log_file = log_path.open('w', encoding='utf-8')
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=environment,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )
        running.append((state, attempt, gpu_id, process, log_file, log_path))

    for state, attempt, gpu_id, process, log_file, log_path in running:
        scene = state['scene']
        key = scene_key(scene)
        model_path = model_path_for(scene)
        train_scene = prepared_scene_paths[key]
        return_code = process.wait()
        log_file.close()

        if return_code == 0:
            trained_scene_settings[key] = attempt
            print(f"DONE {key} on GPU {gpu_id} with {attempt['name']}. Log: {log_path}")
            if Path(train_scene).resolve().is_relative_to(CONVERTED_ROOT.resolve()):
                print('Clean converted training copy:', train_scene)
                shutil.rmtree(train_scene, ignore_errors=True)
            continue

        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(
            f"FAILED {key} on GPU {gpu_id} with {attempt['name']} "
            f"exit={return_code}. Log: {log_path}"
        )
        print('\n'.join(lines[-120:]))
        failed_resume_iter, failed_resume_checkpoint = latest_resume_checkpoint(model_path)
        if AUTO_RESUME:
            print(f'Resume run failed; checkpoint preserved: iteration {failed_resume_iter} at {failed_resume_checkpoint}')
            print('Rerun this training cell to resume instead of deleting the model directory.')
            failed_scenes[key] = log_path
            continue
        next_attempt_idx = state['attempt_idx'] + 1
        if next_attempt_idx < len(TRAIN_ATTEMPTS):
            print('Queue fallback attempt:', TRAIN_ATTEMPTS[next_attempt_idx]['name'], 'for', key)
            train_queue.append({'scene': scene, 'attempt_idx': next_attempt_idx})
        else:
            failed_scenes[key] = log_path

if failed_scenes:
    # ANTIGRAVITY: Thay vi nem loi lam hong Commit, ta chi in canh bao. 
    # Viec nay giup Kaggle luu lai tat ca checkpoints cua cac scene da chay thanh cong truoc khi bi loi.
    print(f'\n[WARNING] Training failed for scenes: {failed_scenes}')
    print('Checkpoints for successful scenes are preserved in outputs.')

print('Batch training complete')
print('Trained scene settings:', trained_scene_settings)


## 7. Kiểm tra đầu ra và dọn dẹp

Kiểm tra các point cloud cuối cùng và xóa repository đã clone khi có thể giải phóng dung lượng Kaggle một cách an toàn.

In [ ]:
expected = []
missing = []
for scene in all_train_scenes:
    marker = trained_model_marker(model_path_for(scene))
    expected.append(marker)
    if not marker.exists():
        missing.append(marker)

print('Expected trained models:', len(expected))
for marker in expected:
    print(marker, 'OK' if marker.exists() else 'MISSING')

if missing:
    raise RuntimeError(f'Missing trained point_cloud.ply files: {missing}')

repo_resolved = REPO_DIR.resolve()
work_resolved = WORK_DIR.resolve()
output_resolved = OUTPUT_ROOT.resolve()
can_cleanup_repo = (
    repo_resolved.name == 'gaussian-splatting'
    and repo_resolved.parent == work_resolved
    and not output_resolved.is_relative_to(repo_resolved)
)
if can_cleanup_repo:
    os.chdir(WORK_DIR)
    print('Remove training repository from Kaggle working directory:', REPO_DIR)
    shutil.rmtree(REPO_DIR)
else:
    print('Skip repository cleanup for safety:', REPO_DIR)

print('Training artifacts are ready in outputs/. Use the inference notebook to render and compute metrics.')
